# P1 · 04 External tests, remedies and DINOv2 reference

Algorithm 7: download of BDNeuro-MRI (Mendeley Data v8) and PMRAM (raw images), removal of every external image with a near-duplicate in the five public sources, external evaluation of all trained models, the remedy comparison (harmonisation, brain mask, Group DRO, SourceShield) and the frozen DINOv2 ViT-S/14 reference (Tables 14, 15, 20, 21, Figs. 8b, 11). **Run after notebook 01.** Needs the `KAGGLE_API_TOKEN` secret. Resume after a disconnect = Run all.

In [ ]:
# Setup: Google Drive, GPU check and code location
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'No GPU: Runtime > Change runtime type > T4 GPU')
# Location of this code package: either a GitHub clone or the folder uploaded to your Drive.
GITHUB_URL = ''   # e.g. the repository URL once released; leave empty to use the Drive copy
CODE_DIR = Path('/content/P1_code_release') if GITHUB_URL else Path('/content/drive/MyDrive/P1_code_release')
if GITHUB_URL and not CODE_DIR.exists():
    subprocess.run(['git', 'clone', '-q', GITHUB_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / 'src' / 'P1_pipeline.py').exists(), f'Code not found in {CODE_DIR}. Upload the P1_code_release folder to MyDrive or set GITHUB_URL.'
SRC = CODE_DIR / 'src'
print('code:', CODE_DIR)
# Kaggle credentials from Colab secrets, passed to the analysis process (values are never printed)
from google.colab import userdata
SECRET_ENV = {}
try:
    tok = userdata.get('KAGGLE_API_TOKEN')
    if tok: SECRET_ENV['KAGGLE_API_TOKEN'] = tok.strip()
except Exception:
    try:
        SECRET_ENV['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); SECRET_ENV['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:
        pass
print('Kaggle credentials found' if SECRET_ENV else 'No Kaggle secret: add KAGGLE_API_TOKEN (key icon, notebook access on)')

In [ ]:
# Run the external tests, remedies and DINOv2 reference. Output is streamed here and written to MyDrive/MI10_P1/. Resume after a disconnect = Run all.
env = dict(os.environ, **SECRET_ENV, P1_MODE='FULL_PAPER_RUN', P1_IN_COLAB='1', P1_ADDON='1', P1_ADDON_PATH=str(SRC / 'P1_addonC.py'))
p = subprocess.Popen([sys.executable, '-u', str(SRC / 'P1_pipeline.py')], env=env, cwd='/content',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='')
p.wait(); print('exit code', p.returncode)